# 1D-CNN — SoH Estimation

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import os
import joblib
import copy

from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

work_dir = os.getcwd()  # should be the Battery_SoH root folder
print("work_dir:", work_dir)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device  :", DEVICE)

MODEL_NAME = "1dcnn"

## Data Loader

In [ ]:
input_folder = os.path.join(work_dir, "data_cnn_normalized")

X_train = np.load(os.path.join(input_folder, "X_train.npy"))
y_train = np.load(os.path.join(input_folder, "y_train.npy"))
X_val = np.load(os.path.join(input_folder, "X_val.npy"))
y_val = np.load(os.path.join(input_folder, "y_val.npy"))
X_test = np.load(os.path.join(input_folder, "X_test.npy"))
y_test = np.load(os.path.join(input_folder, "y_test.npy"))

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"y_train: [{y_train.min():.2f}, {y_train.max():.2f}]")


def create_loader(X, y, batch_size=128, shuffle=True):
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(y, dtype=torch.float32)
    return DataLoader(TensorDataset(Xt, yt), batch_size=batch_size, shuffle=shuffle)


train_loader = create_loader(X_train, y_train)
val_loader = create_loader(X_val, y_val, shuffle=False)
test_loader = create_loader(X_test, y_test, shuffle=False)


## Model

In [ ]:
class CNN1D(nn.Module):
    def __init__(self, input_channels, base_channels=64, dropout=0.2):
        super().__init__()
        bc = base_channels
        self.net = nn.Sequential(
            nn.Conv1d(input_channels, bc, 7, padding=3),
            nn.BatchNorm1d(bc),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(bc, bc * 2, 5, padding=2),
            nn.BatchNorm1d(bc * 2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(bc * 2, bc * 4, 3, padding=1),
            nn.BatchNorm1d(bc * 4),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(bc * 4, bc * 2, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(bc * 2, 1)

    def forward(self, x):
        x = x.permute(0, 2, 1)  # (B, C, T)
        x = self.net(x).squeeze(-1)
        return self.fc(self.dropout(x)).squeeze(-1)


CNN1D_MODEL_CLASS = CNN1D  # alias for CV block

model = CNN1D(input_channels=X_train.shape[2]).to(DEVICE)
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"1D-CNN params: {total:,}")


## Config

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.HuberLoss(delta=3.0)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5, min_lr=1e-6
)
os.makedirs(os.path.join(work_dir, "model"), exist_ok=True)


## Training

In [ ]:
EPOCHS = 300
PATIENCE = 20

train_losses, val_losses = [], []
best_val = float("inf")
best_state = copy.deepcopy(model.state_dict())
counter = 0

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    batch_losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        batch_losses.append(loss.item())
    train_loss = np.mean(batch_losses)

    # Validate
    model.eval()
    batch_losses = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            batch_losses.append(criterion(model(xb), yb).item())
    val_loss = np.mean(batch_losses)

    scheduler.step(val_loss)
    train_losses.append(np.sqrt(train_loss))
    val_losses.append(np.sqrt(val_loss))

    print(
        f"Ep {epoch:03d}/{EPOCHS} | train={np.sqrt(train_loss):.4f} | val={np.sqrt(val_loss):.4f}"
    )

    # Best-epoch checkpoint
    if val_loss < best_val:
        best_val = val_loss
        best_state = copy.deepcopy(model.state_dict())
        counter = 0
        torch.save(best_state, os.path.join(work_dir, "model", f"{'1dcnn'}_best.pt"))
    else:
        counter += 1

    if counter >= PATIENCE:
        print(f"Early stopping at epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"\nTraining done. Best val loss: {best_val:.6f}")


## Evaluate

In [ ]:
model.eval()
preds = []
with torch.no_grad():
    for xb, _ in test_loader:
        preds.append(model(xb.to(DEVICE)).cpu().numpy())
y_pred = np.concatenate(preds)


## Metrics & Figures

In [ ]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE : {rmse:.4f} %")
print(f"MAE  : {mae:.4f} %")
print(f"R²   : {r2:.6f}")

output_figure = os.path.join(work_dir, "figures", "1dcnn")
os.makedirs(output_figure, exist_ok=True)
residuals = y_test - y_pred

# 1 ── True vs Predicted
plt.figure(figsize=(6, 6))
plt.scatter(y_test, y_pred, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
plt.xlabel("True SoH")
plt.ylabel("Predicted SoH")
plt.title("Prediction vs Ground Truth")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig1_true_vs_pred.png"))
plt.show()

# 2 ── Residual plot
plt.figure(figsize=(6, 4))
plt.scatter(y_pred, residuals, alpha=0.3)
plt.axhline(0, color="r")
plt.xlabel("Predicted")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig2_residuals.png"))
plt.show()

# 3 ── Residual distribution
plt.figure(figsize=(6, 4))
sns.histplot(residuals, bins=50, kde=True)
plt.title("Residual Distribution")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig3_residual_dist.png"))
plt.show()

# 4 ── Error vs SoH
plt.figure(figsize=(6, 4))
plt.scatter(y_test, np.abs(residuals), alpha=0.3)
plt.xlabel("True SoH")
plt.ylabel("Absolute Error")
plt.title("Error vs SoH")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig4_error_vs_soh.png"))
plt.show()

# 5 ── Sorted prediction
idx = np.argsort(y_test)
plt.figure(figsize=(8, 4))
plt.plot(y_test[idx], label="True")
plt.plot(y_pred[idx], label="Predicted", alpha=0.6)
plt.legend()
plt.title("Sorted SoH Prediction")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig5_sorted_pred.png"))
plt.show()

# 6 ── Training curve
plt.figure(figsize=(6, 4))
plt.plot(train_losses, label="Train RMSE")
plt.plot(val_losses, label="Val RMSE")
plt.legend()
plt.title(f"Training Curve ({'1dcnn'})")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.grid()
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig6_train_curve.png"))
plt.show()

# 7 ── Violin + box
plt.figure(figsize=(8, 5))
sns.violinplot(x=np.abs(residuals), inner="quartile")
sns.boxplot(x=np.abs(residuals), width=0.2, color="white")
plt.title("Error Distribution (Violin + Box)")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig7_violin_box.png"))
plt.show()

# 8 ── CDF
errors = np.abs(residuals)
sorted_err = np.sort(errors)
cdf = np.arange(len(errors)) / len(errors)
plt.figure()
plt.plot(sorted_err, cdf)
plt.xlabel("Absolute Error")
plt.ylabel("CDF")
plt.title("Error CDF")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig8_cdf.png"))
plt.show()

# 9 ── Histogram + KDE
plt.figure(figsize=(8, 5))
sns.histplot(np.abs(residuals), bins=50, kde=True)
plt.title("Error Distribution (Histogram + KDE)")
plt.xlabel("Absolute Error")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig9_histogram.png"))
plt.show()

# 10 ── Log-scale
plt.figure(figsize=(8, 5))
sns.histplot(np.abs(residuals), bins=50, log_scale=(False, True))
plt.title("Error Distribution (Log Scale)")
plt.xlabel("Absolute Error")
plt.tight_layout()
plt.savefig(os.path.join(output_figure, "Fig10_logscale.png"))
plt.show()

print("MAE    :", np.mean(np.abs(residuals)))
print("RMSE   :", np.sqrt(np.mean(residuals**2)))
print("Max Err:", np.max(np.abs(residuals)))
print("Median :", np.median(np.abs(residuals)))


## Save

In [ ]:
model_dir = os.path.join(work_dir, "model")
os.makedirs(model_dir, exist_ok=True)
joblib.dump(model, os.path.join(model_dir, f"{'1dcnn'}.pkl"))
print(f"Model saved → {model_dir}/{'1dcnn'}.pkl")
